# Optional: a toy signal in the frequency domain
**Extension after the core course, about 20 minutes.**
A small gravitational-wave-inspired exercise without installing a GW stack.
The synthetic sinusoid is not a physical waveform or a detection claim.
You will use sampling units and a known injected frequency to check an FFT.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from astropy import units as u
from astropy.table import QTable
# The Hub pulls the course into your workspace before starting this notebook.
# Prefer that editable checkout; the local editable install is a fallback.
import sys
_candidates = [Path.cwd(), *Path.cwd().parents, Path.home() / "jupyter_data_languages"]
for _candidate in _candidates:
    if (_candidate / "course2026/__init__.py").is_file() and (_candidate / "data/mock_galaxies.ecsv").is_file():
        sys.path.insert(0, str(_candidate))
        break
import course2026
ROOT = Path(course2026.__file__).resolve().parents[1]
plt.rcParams.update({'figure.figsize': (7, 4), 'font.size': 11})

In [ ]:
rng = np.random.default_rng(14)
sample_rate = 512 * u.Hz
duration = 2 * u.s
n_samples = int((sample_rate * duration).to_value(u.dimensionless_unscaled))
time = np.arange(n_samples) / sample_rate.to_value(u.Hz)
injected_frequency = 40.0
signal = np.sin(2 * np.pi * injected_frequency * time) + rng.normal(0, 0.5, n_samples)
frequency = np.fft.rfftfreq(n_samples, d=1 / sample_rate.to_value(u.Hz))
amplitude = 2 * np.abs(np.fft.rfft(signal)) / n_samples
peak = frequency[1:][np.argmax(amplitude[1:])]
resolution = 1 / duration.to_value(u.s)
assert abs(peak - injected_frequency) <= resolution
print('Peak [Hz]:', peak, '| bin spacing [Hz]:', resolution)
fig, ax = plt.subplots()
ax.plot(frequency, amplitude)
ax.set(xlabel='Frequency [Hz]', ylabel='One-sided amplitude (arbitrary units)', xlim=(0, 100))
fig.tight_layout()
plt.show()

### Your turn
1. Halve the duration: predict the bin spacing before rerunning.
2. Try 40.3 Hz: explain why the power spreads between bins (spectral leakage).
3. Try a frequency above the 256 Hz Nyquist frequency: predict aliasing.

The simple factor of two applies to interior positive-frequency bins; DC and
Nyquist bins require different normalisation. Real GW analysis also needs noise
models, conditioning, windowing, calibrated strain and a significance procedure.
An FFT peak alone does not establish an astrophysical event.